# 04 · LoRA

## Opsætning

In [ ]:
# Find training/ (the folder with pyproject.toml), wherever the notebook runs from.
import sys
from pathlib import Path

TRAINING = next(c for p in [Path.cwd(), *Path.cwd().parents] for c in (p, p / "training")
                if (c / "pyproject.toml").is_file() and (c / "wt_train").is_dir())
if str(TRAINING) not in sys.path:
    sys.path.insert(0, str(TRAINING))
print(TRAINING)

In [ ]:
import copy
import logging
import math
import os
import random
import textwrap
import time
import warnings
from html import escape

from wt_train import audio, config, fleurs, paths

# Downloads and caches stay in the work folder; Edda is read from its local snapshot.
os.environ.setdefault("HF_HOME", str(paths.hf_home()))
os.environ["HF_HUB_OFFLINE"] = "1"
# Two harmless start-up notes (torch on macOS, tqdm without widgets) are hidden.
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").addFilter(lambda r: False)
warnings.filterwarnings("ignore", message="IProgress not found")

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import transformers
from IPython.display import HTML, Audio, display
from peft import LoraConfig, get_peft_model
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from transformers.modeling_outputs import BaseModelOutput

transformers.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"


def da(x, digits=0):
    """A number written the Danish way: 2.129.920 and 0,26."""
    return f"{x:,.{digits}f}".replace(",", "_").replace(".", ",").replace("_", ".")


def say(label, text, width=100):
    """`label` and a long text, wrapped so the lines line up under each other."""
    print(textwrap.fill(text or "(tom)", width, initial_indent=f"  {label:<15}", subsequent_indent=" " * 17))


def da_e(x):
    """A tiny number: 7,4e-09, or "præcis 0"."""
    return "præcis 0" if x == 0 else f"{x:.1e}".replace(".", ",")


def table(header, rows, right=()):
    """A small HTML table; columns in `right` are right-aligned."""
    def cell(tag, i, v):
        align = ' style="text-align:right"' if i in right else ' style="text-align:left"'
        return f"<{tag}{align}>{escape(str(v))}</{tag}>"
    head = "".join(cell("th", i, h) for i, h in enumerate(header))
    body = "".join("<tr>" + "".join(cell("td", i, v) for i, v in enumerate(r)) + "</tr>" for r in rows)
    display(HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))


print(f"torch {torch.__version__}, transformers {transformers.__version__}; enhed: {DEVICE}")

## Edda og forsøgets indstillinger

In [ ]:
cfg = config.load_config(paths.CONFIGS / "first-frozen-encoder.yaml")
lora, train_cfg = cfg.lora, cfg.training
print(f"forsøg:        {cfg.name}")
print(f"basismodel:    {cfg.base_model.repo} @ {cfg.base_model.revision[:12]}")
print(f"LoRA:          rank {lora.rank}, alpha {lora.alpha:g}, dropout {da(lora.dropout, 2)}")
print(f"dekoderdele:   {', '.join(lora.decoder)}")
print(f"projektioner:  {', '.join(lora.projections)}")
print(f"koderlag:      {lora.encoder_top_layers} tilpasses (koderen er frosset)")
schedule = {"linear": "lineært ned til 0", "cosine": "som en cosinus ned til 0", "constant": "konstant"}
print(f"træning:       læringsrate {da(train_cfg.learning_rate, 4)}, opvarmning i "
      f"{train_cfg.warmup_fraction * 100:.0f} % af skridtene, derefter {schedule[train_cfg.schedule]}; "
      f"{train_cfg.batch_size} klip x {train_cfg.gradient_accumulation} pr. skridt")

In [ ]:
snapshot = paths.snapshot_dir(cfg.base_model.repo, cfg.base_model.revision)
assert (snapshot / "model.safetensors").is_file(), f"Edda mangler i {snapshot}"
processor = WhisperProcessor.from_pretrained(snapshot)
started = time.time()
# float32 on the Apple GPU, as train.py trains.
model = WhisperForConditionalGeneration.from_pretrained(snapshot, dtype=torch.float32)
model.config.apply_spec_augment = False  # no SpecAugment: the workbench trains on fixed encoder outputs
model = model.to(DEVICE).eval()

n_encoder = sum(p.numel() for p in model.model.encoder.parameters())
n_decoder = sum(p.numel() for p in model.model.decoder.parameters())
n_edda = sum(p.numel() for p in model.parameters())
print(f"indlæst fra {snapshot} på {time.time() - started:.0f} s")
print(f"koder:   {model.config.encoder_layers} lag, {da(n_encoder)} parametre")
print(f"dekoder: {model.config.decoder_layers} lag, {da(n_decoder)} parametre (ordforrådet deles med udgangslaget)")
print(f"i alt:   {da(n_edda)} parametre")

## Klip fra FLEURS

In [ ]:
N_TRAIN = 48  # training clips: 30 steps of 16 clips go 10 times through them
clips = [c for c in fleurs.load_split("train") if c.seconds <= 30]  # Whisper hears 30 s at a time
random.Random(train_cfg.seed).shuffle(clips)
train_clips = clips[:N_TRAIN]
seen = {c.sentence_id for c in train_clips}
held_out = [c for c in clips[N_TRAIN:] if c.sentence_id not in seen][:2]
# Four clips to compare texts on: two it trains on, two it never sees.
check_clips = train_clips[:2] + held_out
check_kind = ["trænet på"] * 2 + ["ikke trænet på"] * 2

print(f"FLEURS da_dk train: {len(clips)} klip; vi træner på {len(train_clips)} "
      f"({da(sum(c.seconds for c in train_clips) / 60, 1)} minutters tale)")
for c, kind in zip(check_clips, check_kind):
    print(f"{c.id} ({kind}, {da(c.seconds, 1)} s)")
    say("", c.text)
display(Audio(audio.read_16k_mono(check_clips[0].wav), rate=audio.SAMPLE_RATE))

## Koderen køres én gang

In [ ]:
def log_mel(batch):
    """Log-mel features [clips, 128, 3000]: 30 s windows, padded with silence."""
    pcms = [audio.read_16k_mono(c.wav) for c in batch]
    return processor.feature_extractor(pcms, sampling_rate=audio.SAMPLE_RATE, return_tensors="pt").input_features


# The encoder is frozen, so its output for a clip never changes: compute it once, in float16
# (Edda's own precision; the weights are float16 on disk, so this copy is exact), as train.py's cache does.
encoder16 = copy.deepcopy(model.model.encoder).to(torch.float16)
cache = {}
started = time.time()
todo = train_clips + held_out
for i in range(0, len(todo), 8):
    batch = todo[i:i + 8]
    with torch.inference_mode():
        hidden = encoder16(log_mel(batch).to(DEVICE, torch.float16)).last_hidden_state
    for c, h in zip(batch, hidden):
        cache[c.id] = h  # [1500, 1280] float16, kept on the GPU
del encoder16
if DEVICE == "mps":
    torch.mps.synchronize()
    torch.mps.empty_cache()

one = next(iter(cache.values()))
print(f"{len(cache)} klip gennem koderen på {time.time() - started:.0f} s")
print(f"pr. klip: {one.shape[0]} x {one.shape[1]} tal i float16 = {da(one.numel() * 2 / 1e6, 2)} MB")
steps, per_step = 30, train_cfg.batch_size * train_cfg.gradient_accumulation
print(f"uden cache skulle koderen køre {steps * per_step} gange i {steps} skridt; nu kører den {len(cache)} gange")

## Tekst som tokens

In [ ]:
tokenizer = processor.tokenizer
PROMPT = tokenizer.convert_tokens_to_ids(["<|startoftranscript|>", "<|da|>", "<|transcribe|>", "<|notimestamps|>"])
EOT = tokenizer.convert_tokens_to_ids("<|endoftext|>")
IGNORE = -100  # cross_entropy skips these targets


def example(text):
    """(decoder input ids, labels), exactly as Edda was trained: the fixed prompt, the text, the end."""
    sequence = PROMPT + tokenizer.encode(text, add_special_tokens=False) + [EOT]
    inputs, labels = sequence[:-1], sequence[1:]  # token i predicts token i + 1
    fixed = len(PROMPT) - 1
    return inputs, [IGNORE] * fixed + labels[fixed:]  # the loss never covers the fixed prompt


examples = {c.id: example(c.text) for c in todo}
inputs, labels = examples[train_clips[0].id]
print(train_clips[0].text)
print(f"{'nr':>3}  {'dekoderen ser':<24} {'og skal gætte':<24}")
for i, (x, y) in enumerate(zip(inputs[:10], labels[:10])):
    target = "(tæller ikke)" if y == IGNORE else repr(tokenizer.decode([y]))
    print(f"{i:>3}  {tokenizer.decode([x])!r:<24} {target:<24}")
print(f"... {len(inputs)} tokens i alt; tabet tæller de {sum(y != IGNORE for y in labels)} sidste")

In [ ]:
def collate(ids):
    """Right-padded (inputs, labels) tensors for a batch of clips."""
    pairs = [examples[i] for i in ids]
    width = max(len(x) for x, _ in pairs)
    inputs = torch.full((len(pairs), width), EOT, dtype=torch.long)
    labels = torch.full((len(pairs), width), IGNORE, dtype=torch.long)
    for row, (x, y) in enumerate(pairs):
        inputs[row, :len(x)] = torch.tensor(x)
        labels[row, :len(y)] = torch.tensor(y)
    return inputs.to(DEVICE), labels.to(DEVICE)


def encoder_out(ids):
    return BaseModelOutput(last_hidden_state=torch.stack([cache[i] for i in ids]).float())


GENERATE = dict(language="da", task="transcribe", return_timestamps=False, do_sample=False, num_beams=1,
                max_new_tokens=cfg.evaluation.max_new_tokens)  # greedy, like the app


def transcribe(m, batch):
    ids = [c.id for c in batch]
    with torch.inference_mode():
        out = m.generate(encoder_outputs=encoder_out(ids), **GENERATE)
    return [tokenizer.decode(row, skip_special_tokens=True).strip() for row in out]


def logits_of(m, batch):
    """The decoder's scores for every token of the reference text (teacher forcing)."""
    ids = [c.id for c in batch]
    inputs, _ = collate(ids)
    with torch.inference_mode():
        return m(encoder_outputs=encoder_out(ids), decoder_input_ids=inputs).logits.float()

## Edda før LoRA

In [ ]:
base_texts = transcribe(model, check_clips)
base_logits = logits_of(model, check_clips)
for c, text in zip(check_clips, base_texts):
    print(c.id)
    say("reference:", c.text)
    say("Edda:", text)

## LoRA på dekoderen

In [ ]:
assert lora.encoder_top_layers == 0, "denne notesbog træner med koderen frosset"
# A PEFT target_modules regex (a full match on module names), built as train.py builds it.
projections = "|".join(lora.projections)
decoder_parts = {
    "self_attn": rf"self_attn\.(?:{projections})",      # attention over the words so far
    "cross_attn": rf"encoder_attn\.(?:{projections})",  # attention over the audio
    "ffn": r"fc1|fc2",                                  # the feed-forward block
}
parts = [rf"model\.decoder\.layers\.\d+\.(?:{'|'.join(decoder_parts[p] for p in lora.decoder)})"]
target_modules = "|".join(f"(?:{p})" for p in parts)

torch.manual_seed(train_cfg.seed)  # lora_A starts random, from the seed
peft_model = get_peft_model(model, LoraConfig(r=lora.rank, lora_alpha=lora.alpha, lora_dropout=lora.dropout,
                                              target_modules=target_modules, bias="none"))
inner = peft_model.get_base_model()  # Edda itself, now with LoRA layers inside
adapted = [name for name, m in inner.named_modules() if hasattr(m, "lora_A") and "default" in m.lora_A]
print(target_modules)
print(f"{len(adapted)} moduler fik en adapter, fx {adapted[0]}")
print(inner.model.decoder.layers[0].encoder_attn.q_proj)

## Trænbare parametre

In [ ]:
part_name = {"self_attn": "selvopmærksomhed", "encoder_attn": "krydsopmærksomhed", "fc1": "fremadkobling",
             "fc2": "fremadkobling"}
kinds = {}
for name in adapted:
    m = inner.get_submodule(name)
    kind = name.split(".layers.")[1].split(".", 1)[1]  # "self_attn.q_proj", "fc1", ...
    a, b = m.lora_A["default"].weight, m.lora_B["default"].weight
    k = kinds.setdefault(kind, {"layers": 0, "w": tuple(m.base_layer.weight.shape), "a": tuple(a.shape),
                                "b": tuple(b.shape), "each": a.numel() + b.numel()})
    k["layers"] += 1

order = ["self_attn", "encoder_attn", "fc1", "fc2"]  # the config's order: self, cross, ffn; q, k, v, out


def place(kind):
    part, _, projection = kind.partition(".")
    return order.index(part), lora.projections.index(projection) if projection else 0


rows = [(kind, part_name[kind.split(".")[0]], k["layers"], f"{k['w'][0]} x {k['w'][1]}",
         f"{k['a'][0]} x {k['a'][1]}", f"{k['b'][0]} x {k['b'][1]}", da(k["each"]), da(k["each"] * k["layers"]))
        for kind, k in sorted(kinds.items(), key=lambda kv: place(kv[0]))]
trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in peft_model.parameters())
rows.append(("i alt", "", "", "", "", "", "", da(trainable)))
table(["modul", "del", "lag", "vægt W (ud x ind)", "A (rank x ind)", "B (ud x rank)", "pr. lag", "i alt"], rows,
      right={2, 6, 7})

assert all("lora_" in n for n, p in peft_model.named_parameters() if p.requires_grad)
table(["del af modellen", "parametre", "trænes"], [
    (f"koder, {model.config.encoder_layers} lag", da(n_encoder), "nej, frosset"),
    (f"dekoder, {model.config.decoder_layers} lag", da(n_decoder), "nej, frosset"),
    ("LoRA-adaptere (A og B)", da(trainable), "ja"),
    ("i alt", da(total), ""),
], right={1})
print(f"Trænbare parametre: {da(trainable)} af {da(total)} = {da(trainable / total * 100, 2)} %")

## lora_B starter i nul

In [ ]:
lora_a = [p for n, p in peft_model.named_parameters() if "lora_A" in n]
lora_b = [p for n, p in peft_model.named_parameters() if "lora_B" in n]
scale = lora.alpha / lora.rank
print(f"{len(lora_b)} lora_B-matricer; største |værdi|: {da_e(max(p.abs().max().item() for p in lora_b))}")
print(f"{len(lora_a)} lora_A-matricer, tilfældige; største |værdi|: {da(max(p.abs().max().item() for p in lora_a), 4)}")
print(f"ny vægt = W + (alpha / rank) x B x A = W + {scale:g} x B x A; med B = 0 er det W selv")

In [ ]:
peft_model.eval()  # no dropout
wrapped_texts = transcribe(inner, check_clips)
wrapped_logits = logits_of(inner, check_clips)
print(f"største forskel i logits, Edda mod Edda med LoRA: {da_e((wrapped_logits - base_logits).abs().max().item())}")
for c, before, after in zip(check_clips, base_texts, wrapped_texts):
    verdict = "samme tekst" if before == after else "FORSKELLIG TEKST"
    print(f"{c.id}: {verdict}")
    say("Edda:", before)
    say("Edda med LoRA:", after)

## Træning i 30 skridt

In [ ]:
STEPS = 30
batch_size, accumulation = train_cfg.batch_size, train_cfg.gradient_accumulation
warmup = round(train_cfg.warmup_fraction * STEPS)


def lr_factor(step):
    """Share of the peak learning rate at optimiser step `step` (from 0), as train.py."""
    if warmup and step < warmup:
        return (step + 1) / warmup
    if train_cfg.schedule == "constant":
        return 1.0
    progress = min(1.0, (step - warmup) / max(1, STEPS - warmup))
    if train_cfg.schedule == "cosine":
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    return max(0.0, 1.0 - progress)


params = [p for p in peft_model.parameters() if p.requires_grad]  # only A and B
optimizer = torch.optim.AdamW(params, lr=train_cfg.learning_rate, weight_decay=train_cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
text_tokens = {i: sum(1 for y in examples[i][1] if y != IGNORE) for i in examples}

fig, (ax, ax_lr) = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [2.2, 1]})
plt.close(fig)  # drawn through the display handle below, updated every step
handle = display(fig, display_id=True)
step_loss, step_lr, epoch_loss = [], [], []


def draw():
    ax.clear()
    ax_lr.clear()
    x = range(1, len(step_loss) + 1)
    ax.plot(x, step_loss, marker="o", markersize=3, linewidth=1, color="#4c6ef5", label="hvert skridt")
    if epoch_loss:
        ax.plot(*zip(*epoch_loss), marker="s", linewidth=2.5, color="#e8590c", label="gennemsnit pr. epoke")
    ax.set(xlim=(0.5, STEPS + 0.5), ylim=(0, None), xlabel="skridt", ylabel="tab pr. teksttoken",
           title=f"Tab under træning ({len(step_loss)} af {STEPS} skridt)")
    ax.legend(loc="upper right")
    ax.grid(alpha=0.3)
    ax_lr.plot(x, step_lr, color="#2b8a3e")
    ax_lr.set(xlim=(0.5, STEPS + 0.5), ylim=(0, train_cfg.learning_rate * 1.1), xlabel="skridt",
              title="Læringsrate")
    ax_lr.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
    ax_lr.grid(alpha=0.3)
    fig.tight_layout()
    handle.update(fig)


torch.manual_seed(train_cfg.seed)
peft_model.train()  # LoRA dropout on
started, step, epoch = time.time(), 0, 0
while step < STEPS:
    epoch += 1
    # A new order of the clips every epoch, seeded as in train.py.
    generator = torch.Generator().manual_seed(train_cfg.seed * 1000 + epoch)
    order = [train_clips[i].id for i in torch.randperm(len(train_clips), generator=generator).tolist()]
    micro = [order[i:i + batch_size] for i in range(0, len(order), batch_size)]
    loss_sum, token_sum = 0.0, 0
    for g in range(0, len(micro), accumulation):
        group = micro[g:g + accumulation]
        group_tokens = sum(text_tokens[i] for batch in group for i in batch)
        group_loss = 0.0
        optimizer.zero_grad(set_to_none=True)
        for batch in group:
            inputs, labels = collate(batch)
            logits = inner(encoder_outputs=encoder_out(batch), decoder_input_ids=inputs).logits
            loss = F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), labels.reshape(-1),
                                   ignore_index=IGNORE, reduction="sum")
            (loss / group_tokens).backward()  # the step's loss is the mean over its text tokens
            group_loss += float(loss.detach())
        if train_cfg.max_grad_norm > 0:
            torch.nn.utils.clip_grad_norm_(params, train_cfg.max_grad_norm)
        step_lr.append(scheduler.get_last_lr()[0])
        optimizer.step()
        scheduler.step()
        step += 1
        loss_sum += group_loss
        token_sum += group_tokens
        step_loss.append(group_loss / group_tokens)
        draw()  # the curve grows while it trains
        if step == STEPS:
            break
    epoch_loss.append((step, loss_sum / token_sum))
    draw()
peft_model.eval()

seconds = time.time() - started
print(f"{STEPS} skridt med {batch_size * accumulation} klip i hvert ({epoch} epoker gennem {len(train_clips)} klip) "
      f"på {seconds:.0f} s, {da(seconds / STEPS, 1)} s pr. skridt")
print(f"tab i første skridt: {da(step_loss[0], 4)}; i sidste skridt: {da(step_loss[-1], 4)}")
print(f"gennemsnit i første epoke: {da(epoch_loss[0][1], 4)}; i sidste epoke: {da(epoch_loss[-1][1], 4)}")
print(f"største |værdi| i lora_B nu: {da(max(p.abs().max().item() for p in lora_b), 4)}")

## Flet adapteren ind

In [ ]:
adapter_texts = transcribe(inner, check_clips)
adapter_logits = logits_of(inner, check_clips)

# One weight before merging, to see what merging does to it.
q = inner.model.decoder.layers[0].encoder_attn.q_proj
w_before = q.base_layer.weight.detach().clone()
update = scale * (q.lora_B["default"].weight @ q.lora_A["default"].weight).detach()

merged = peft_model.merge_and_unload()  # W <- W + (alpha / rank) x B x A, then the LoRA layers go
w_after = merged.model.decoder.layers[0].encoder_attn.q_proj.weight.detach()
print(merged.model.decoder.layers[0].encoder_attn.q_proj)
print(f"LoRA-moduler tilbage: {sum(1 for n, _ in merged.named_modules() if 'lora' in n)}")
print(f"parametre efter fletning: {da(sum(p.numel() for p in merged.parameters()))} (Edda: {da(n_edda)})")
print(f"decoder.layers.0.encoder_attn.q_proj: største |ændring| af W: {da(update.abs().max().item(), 6)}; "
      f"W efter - W før - {scale:g} x B x A: højst {da_e((w_after - w_before - update).abs().max().item())}")

## Samme tekst efter fletning

In [ ]:
merged_texts = transcribe(merged, check_clips)
merged_logits = logits_of(merged, check_clips)
diff = (merged_logits - adapter_logits).abs().max().item()


rows_html, same = [], 0
for c, kind, before, adapted_text, merged_text in zip(check_clips, check_kind, base_texts, adapter_texts,
                                                       merged_texts):
    ok = adapted_text == merged_text
    same += ok
    verdict = "samme tekst" if ok else "FORSKELLIG TEKST"
    color = "#2b8a3e" if ok else "#c92a2a"
    changed = "ja" if adapted_text != before else "nej"
    rows_html.append(
        f"<tr><td style='text-align:left;vertical-align:top'>{escape(c.id)}<br><small>{kind}</small></td>"
        f"<td style='text-align:left;vertical-align:top'>{escape(adapted_text) or '<i>(tom)</i>'}</td>"
        f"<td style='text-align:left;vertical-align:top'>{escape(merged_text) or '<i>(tom)</i>'}</td>"
        f"<td style='text-align:left;vertical-align:top;white-space:nowrap'><b style='color:{color}'>{verdict}</b>"
        f"<br><small>ændret af træningen: {changed}</small></td></tr>")
display(HTML("<table><thead><tr><th style='text-align:left'>klip</th><th style='text-align:left'>med adapter</th>"
             "<th style='text-align:left'>flettet</th><th style='text-align:left'>dom</th></tr></thead>"
             f"<tbody>{''.join(rows_html)}</tbody></table>"))
print(f"{same} af {len(check_clips)} klip: samme tekst; største forskel i logits: {da_e(diff)}")

In [ ]:
# Nothing was written: the adapter and the merged model lived only in memory.
del peft_model, inner, merged, model, cache, base_logits, wrapped_logits, adapter_logits, merged_logits
if DEVICE == "mps":
    torch.mps.empty_cache()
print("Færdig. Intet er gemt: adapteren og den flettede model fandtes kun i hukommelsen.")